## EXP-GT-004 — Freeze Ground-Truth Benchmark

### Purpose

I have validated the source artifact, knowledge fragment, annotation, provenance, and source integrity.

I will now freeze them into a single versioned benchmark so future CEREBRO experiments can use the same trusted ground truth without depending on another notebook's memory.

### Load the validated components

In [8]:
import json
from pathlib import Path

ground_truth_dir = Path("../data/ground_truth")

artifact_path = ground_truth_dir / "artifact.json"
knowledge_fragment_path = ground_truth_dir / "knowledge_fragment.json"
annotation_path = ground_truth_dir / "annotation.json"

with artifact_path.open("r", encoding="utf-8") as f:
    artifact = json.load(f)

with knowledge_fragment_path.open("r", encoding="utf-8") as f:
    knowledge_fragment = json.load(f)

with annotation_path.open("r", encoding="utf-8") as f:
    annotation = json.load(f)

print("✓ Artifact:", artifact["artifact_id"])
print("✓ Knowledge Fragment:", knowledge_fragment["fragment_id"])
print("✓ Annotation:", annotation["annotation_id"])

✓ Artifact: ART-0001
✓ Knowledge Fragment: KF-0001
✓ Annotation: ANN-0001


### Final pre-freeze integrity check

In [9]:
assert artifact["artifact_id"] == knowledge_fragment["artifact_id"]

assert artifact["artifact_id"] == annotation["artifact_id"]

assert knowledge_fragment["fragment_id"] == annotation["fragment_id"]

assert artifact["source"]["sha256"], (
    "Artifact SHA-256 has not been registered."
)

assert annotation["manually_verified"] is True, (
    "Ground truth has not been manually verified."
)

print("✓ Cross-references verified")
print("✓ Source checksum registered")
print("✓ Ground truth manually verified")
print("\nBenchmark ready to freeze.")

✓ Cross-references verified
✓ Source checksum registered
✓ Ground truth manually verified

Benchmark ready to freeze.


### Create the frozen benchmark

In [10]:
benchmark = {
    "benchmark_id": "CEREBRO-GT-v0.1",
    "experiment_id": "EXP-GT-004",
    "status": "frozen",

    "ground_truth_method": "manual_verification",

    "integrity_rules": [
        "CIF-01",
        "CIF-02"
    ],

    "artifacts": [
        artifact
    ],

    "knowledge_fragments": [
        knowledge_fragment
    ],

    "annotations": [
        annotation
    ]
}

print("Benchmark :", benchmark["benchmark_id"])
print("Status    :", benchmark["status"])
print("Artifacts :", len(benchmark["artifacts"]))
print("Fragments :", len(benchmark["knowledge_fragments"]))
print("Annotations:", len(benchmark["annotations"]))

Benchmark : CEREBRO-GT-v0.1
Status    : frozen
Artifacts : 1
Fragments : 1
Annotations: 1


### Persist the frozen benchmark

In [11]:
benchmark_path = (
    ground_truth_dir / "CEREBRO-GT-v0.1.json"
)

with benchmark_path.open("w", encoding="utf-8") as f:
    json.dump(
        benchmark,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ Frozen benchmark persisted")
print("Path:", benchmark_path)

✓ Frozen benchmark persisted
Path: ../data/ground_truth/CEREBRO-GT-v0.1.json


Reload from disk

In [12]:
with benchmark_path.open("r", encoding="utf-8") as f:
    frozen_benchmark = json.load(f)

assert frozen_benchmark["benchmark_id"] == "CEREBRO-GT-v0.1"
assert frozen_benchmark["status"] == "frozen"

assert len(frozen_benchmark["artifacts"]) == 1
assert len(frozen_benchmark["knowledge_fragments"]) == 1
assert len(frozen_benchmark["annotations"]) == 1

print("✓ Frozen benchmark successfully reloaded")

✓ Frozen benchmark successfully reloaded


### Verify frozen references

In [13]:
frozen_artifact = frozen_benchmark["artifacts"][0]
frozen_fragment = frozen_benchmark["knowledge_fragments"][0]
frozen_annotation = frozen_benchmark["annotations"][0]

assert (
    frozen_fragment["artifact_id"]
    == frozen_artifact["artifact_id"]
)

assert (
    frozen_annotation["artifact_id"]
    == frozen_artifact["artifact_id"]
)

assert (
    frozen_annotation["fragment_id"]
    == frozen_fragment["fragment_id"]
)

assert frozen_artifact["source"]["sha256"]

print("✓ Artifact reference verified")
print("✓ Knowledge fragment reference verified")
print("✓ Annotation reference verified")
print("✓ Source integrity fingerprint preserved")

✓ Artifact reference verified
✓ Knowledge fragment reference verified
✓ Annotation reference verified
✓ Source integrity fingerprint preserved


### Final Step 01 acceptance

In [14]:
print("CEREBRO Ground-Truth Benchmark")
print("------------------------------")
print("Benchmark :", frozen_benchmark["benchmark_id"])
print("Status    :", frozen_benchmark["status"])
print("Artifact  :", frozen_artifact["artifact_id"])
print("Fragment  :", frozen_fragment["fragment_id"])
print("Annotation:", frozen_annotation["annotation_id"])
print("SHA-256   :", frozen_artifact["source"]["sha256"])

print("\n✓ CEREBRO-GT-v0.1 FROZEN")
print("✓ STEP 01 COMPLETE")

CEREBRO Ground-Truth Benchmark
------------------------------
Benchmark : CEREBRO-GT-v0.1
Status    : frozen
Artifact  : ART-0001
Fragment  : KF-0001
Annotation: ANN-0001
SHA-256   : 7694567acb26611218fd818ef2c12ec456e5ee94c5387804bfe5f36799aee832

✓ CEREBRO-GT-v0.1 FROZEN
✓ STEP 01 COMPLETE


### Result

CEREBRO-GT-v0.1 is now frozen as my first trusted ground-truth benchmark.

Future experiments will compare CEREBRO's automated outputs against this benchmark rather than relying on notebook state or manually recreated reference data.

Any change to the benchmark should create a new version rather than silently modifying this frozen baseline.

flowchart LR
    GT[CEREBRO-GT-v0.1<br/>Frozen Ground Truth] --> TI[02 Text Ingestion]
    TI --> PO[Pipeline Output]
    PO --> EV[Evaluation]
    GT --> EV
    EV --> M[Measured Accuracy]